In [1]:
import pandas as pd, numpy as np
from scipy import stats

data = pd.read_csv("../data/zomato_ncr_clean.csv")
rated = data[data["is_rated"]]

def compare(a, b, label):
    t, p = stats.ttest_ind(a, b, equal_var=False)   # Welch t-test
    print(f"{label}: {a.mean():.2f} vs {b.mean():.2f} | diff {a.mean()-b.mean():+.2f} | n={len(a)}/{len(b)} | p={p:.2e}")

In [2]:
yes = rated[rated["has_table_booking"] == "Yes"]["aggregate_rating"]
no  = rated[rated["has_table_booking"] == "No"]["aggregate_rating"]
compare(yes, no, "Table booking")

Table booking: 3.51 vs 3.23 | diff +0.28 | n=1002/4806 | p=9.84e-56


In [3]:
for pr in [2, 3, 4]:
    s = rated[rated["price_range"] == pr]
    compare(s[s["has_table_booking"] == "Yes"]["aggregate_rating"],
            s[s["has_table_booking"] == "No"]["aggregate_rating"],
            f"Price range {pr}")

Price range 2: 3.30 vs 3.24 | diff +0.06 | n=202/1998 | p=9.76e-02
Price range 3: 3.54 vs 3.62 | diff -0.08 | n=559/224 | p=6.07e-02
Price range 4: 3.62 vs 3.78 | diff -0.16 | n=240/54 | p=3.31e-02


In [4]:
compare(rated[rated["has_online_delivery"] == "Yes"]["aggregate_rating"],
        rated[rated["has_online_delivery"] == "No"]["aggregate_rating"],
        "Online delivery")

Online delivery: 3.34 vs 3.25 | diff +0.09 | n=2227/3581 | p=5.13e-12


In [5]:
compare(rated[rated["is_chain"]]["aggregate_rating"],
        rated[~rated["is_chain"]]["aggregate_rating"], "Chain vs independent")

for pr in [1, 2, 3]:
    s = rated[rated["price_range"] == pr]
    compare(s[s["is_chain"]]["aggregate_rating"],
            s[~s["is_chain"]]["aggregate_rating"], f"Price range {pr}")

Chain vs independent: 3.17 vs 3.31 | diff -0.13 | n=1099/4709 | p=1.45e-18
Price range 1: 3.17 vs 3.18 | diff -0.02 | n=486/2045 | p=3.84e-01
Price range 2: 3.16 vs 3.27 | diff -0.12 | n=532/1668 | p=9.89e-07
Price range 3: 3.30 vs 3.60 | diff -0.30 | n=81/702 | p=1.13e-05


In [6]:
r1, p1 = stats.spearmanr(rated["votes"], rated["aggregate_rating"])
r2, p2 = stats.spearmanr(rated["average_cost_for_two"], rated["aggregate_rating"])
print(f"Votes vs rating: rho={r1:.2f}")
print(f"Cost vs rating:  rho={r2:.2f}")

Votes vs rating: rho=0.59
Cost vs rating:  rho=0.27


In [7]:
groups = [g["aggregate_rating"] for c, g in rated.groupby("city") if len(g) >= 100]
print(stats.f_oneway(*groups))

F_onewayResult(statistic=np.float64(28.45655198363385), pvalue=np.float64(2.932393758827249e-18))
